In [1]:
# PERFORM A BASIC SANITY CHECK ON RAG, INGESTOR URLs

import requests

RAG_URL = "http://rag-server:8081"
INGESTOR_URL = "http://ingestor-server:8082"

rag_health = requests.get(
    f"{RAG_URL}/v1/health",
    params={"check_dependencies": "true"},
    timeout=30
)

ingestor_health = requests.get(
    f"{INGESTOR_URL}/v1/health",
    params={"check_dependencies": "true"},
    timeout=30
)

print("RAG:", rag_health.status_code)
print(rag_health.text)

print("Ingestor:", ingestor_health.status_code)
print(ingestor_health.text)

RAG: 200
{"message":"Service is up.","databases":[{"service":"Elasticsearch","url":"http://rag-eck-elasticsearch-es-default:9200","status":"healthy","latency_ms":14.61,"error":null,"collections":null}],"object_storage":[{"service":"Object Storage","url":"rag-seaweedfs-all-in-one:9010","status":"healthy","latency_ms":20.85,"error":null,"buckets":1,"message":null}],"nim":[{"service":"LLM","url":"http://nim-llm:8000/v1/health/ready","status":"healthy","latency_ms":5.34,"error":null,"model":"nvidia/nemotron-3-super-120b-a12b","message":null,"http_status":200},{"service":"Embeddings","url":"http://nemotron-vlm-embedding-ms:8000/v1/health/ready","status":"healthy","latency_ms":5.24,"error":null,"model":"nvidia/llama-nemotron-embed-vl-1b-v2","message":null,"http_status":200},{"service":"Ranking","url":"http://nemotron-ranking-ms:8000/v1/health/ready","status":"healthy","latency_ms":12.53,"error":null,"model":"nvidia/llama-nemotron-rerank-1b-v2","message":null,"http_status":200}]}
Ingestor: 20

# CREATE A NEW COLLECTION IN THE INGESTOR


The current Ingestor API exposes:
- POST /v1/collection — create a collection
- GET /v1/collections — list collections
- POST /v1/documents — ingest documents into a specified collection
- GET /v1/status — check ingestion status

In [9]:
# Ingestor URL and Vector DB endpoints:

INGESTOR_URL = "http://ingestor-server:8082/v1"
VDB_ENDPOINT = "http://rag-eck-elasticsearch-es-default:9200"

In [10]:
# Check the existing collections:

response = requests.get(
    f"{INGESTOR_URL}/collections",
    timeout=30
)

print("HTTP Status:", response.status_code)
print(response.json())

HTTP Status: 200
{'message': 'Collections listed successfully.', 'total_collections': 1, 'collections': [{'collection_name': 'multimodal_data', 'num_entities': 58, 'metadata_schema': [{'name': 'filename', 'type': 'string', 'required': False, 'array_type': None, 'max_length': None, 'description': 'Name of the uploaded file'}], 'collection_info': {'has_charts': True, 'doc_type_counts': {'table': 19, 'chart': 6, 'text': 33}, 'last_indexed': '2026-10-04T06:11:48.095975+00:00', 'raw_text_elements_size': 97297, 'total_elements': 58, 'last_ingestion_error': '', 'has_images': False, 'number_of_files': 2, 'has_tables': True, 'ingestion_status': 'Success', 'description': 'Demo collection for NVIDIA RAG Blueprint', 'tags': ['rag', 'demo'], 'owner': 'jupyter', 'created_by': '', 'business_domain': '', 'status': 'Active', 'date_created': '2026-09-28T06:29:53.117742+00:00', 'last_updated': '2026-09-28T06:29:53.117742+00:00'}}]}


In [11]:
# Create a new collection
collection_name = "Research_Papers"

payload = {
    "vdb_endpoint": VDB_ENDPOINT,
    "collection_name": collection_name,
    "description": "Knowledge base for research papers on LLMs",
    "tags": [
        "ai",
        "llm",
        "research",
        "blt",
        "slm"
    ],
    "metadata_schema": []
}

response = requests.post(
    f"{INGESTOR_URL}/collection",
    json=payload,
    timeout=60
)

print("HTTP Status:", response.status_code)
print(response.json())

HTTP Status: 200
{'message': 'Collection research_papers created successfully.', 'collection_name': 'research_papers'}


In [12]:
# Check the newly created collections:

response = requests.get(
    f"{INGESTOR_URL}/collections",
    timeout=30
)

print("HTTP Status:", response.status_code)
print(response.json())

HTTP Status: 200
{'message': 'Collections listed successfully.', 'total_collections': 2, 'collections': [{'collection_name': 'multimodal_data', 'num_entities': 58, 'metadata_schema': [{'name': 'filename', 'type': 'string', 'required': False, 'array_type': None, 'max_length': None, 'description': 'Name of the uploaded file'}], 'collection_info': {'has_charts': True, 'doc_type_counts': {'table': 19, 'chart': 6, 'text': 33}, 'last_indexed': '2026-10-04T06:11:48.095975+00:00', 'raw_text_elements_size': 97297, 'total_elements': 58, 'last_ingestion_error': '', 'has_images': False, 'number_of_files': 2, 'has_tables': True, 'ingestion_status': 'Success', 'description': 'Demo collection for NVIDIA RAG Blueprint', 'tags': ['rag', 'demo'], 'owner': 'jupyter', 'created_by': '', 'business_domain': '', 'status': 'Active', 'date_created': '2026-09-28T06:29:53.117742+00:00', 'last_updated': '2026-09-28T06:29:53.117742+00:00'}}, {'collection_name': 'research_papers', 'num_entities': 0, 'metadata_schema

In [13]:
from pathlib import Path

PDF_DIRECTORY = Path("/home/jovyan/work/reasearch_papers/")

pdf_files = sorted(
    p for p in PDF_DIRECTORY.iterdir()
    if p.is_file() and p.suffix.lower() == ".pdf"
)

print(f"PDF directory: {PDF_DIRECTORY}")
print(f"PDF files found: {len(pdf_files)}")

for pdf in pdf_files:
    print(f" - {pdf.name}")

PDF directory: /home/jovyan/work/reasearch_papers
PDF files found: 2
 - BLT-Paper.pdf
 - SLM_Future_AI.pdf


In [16]:
import json
import requests
from pathlib import Path

INGESTOR_URL = "http://ingestor-server:8082/v1"

metadata = {
    "collection_name": "research_papers",
    "blocking": False,
    "split_options": {
        "chunk_size": 512,
        "chunk_overlap": 150
    },
    "custom_metadata": [],
    "generate_summary": False,
    "documents_catalog_metadata": [],
    "summary_options": None,
    "enable_pdf_split_processing": False,
    "pdf_split_processing_options": {
        "pages_per_chunk": 16
    }
}

ingestion_tasks = {}

for file_path in pdf_files:
    print(f"Uploading: {file_path.name}")
    with open(file_path, "rb") as f:
        response = requests.post(
            f"{INGESTOR_URL}/documents",
            files={
                "documents": (
                    file_path.name,
                    f,
                    "application/pdf"
                )
            },
            data={
                "data": json.dumps(metadata)
            },
            timeout=120
        )
        ingestion_tasks[file_path.name] = task_id

    print("HTTP Status:", response.status_code)
    print(response.text)
    

Uploading: BLT-Paper.pdf
HTTP Status: 200
{"message":"Ingestion started in background","task_id":"8921de40-94ac-4e4c-aac9-c58a593a1095"}
Uploading: SLM_Future_AI.pdf
HTTP Status: 200
{"message":"Ingestion started in background","task_id":"4f63f701-ce6b-4669-94a4-94041228008c"}


In [18]:
ingestion_tasks

{'BLT-Paper.pdf': '24744340-a02d-43ab-a0d4-f4b2b34318ea',
 'SLM_Future_AI.pdf': '24744340-a02d-43ab-a0d4-f4b2b34318ea'}

In [20]:
import requests

for file, task_id in ingestion_tasks.items():
    
    status_response = requests.get(
        f"{INGESTOR_URL}/status",
        params={"task_id": task_id},
        timeout=30
    )

    print("HTTP Status:", status_response.status_code)
    print(status_response.json())

HTTP Status: 200
{'state': 'FINISHED', 'result': {'message': 'Document upload job successfully completed.', 'total_documents': 1, 'documents_completed': 1, 'batches_completed': 1, 'documents': [{'document_name': 'SLM_Future_AI.pdf', 'metadata': {'filename': 'SLM_Future_AI.pdf'}, 'document_info': {'description': '', 'tags': [], 'document_type': 'pdf', 'file_size': 269920, 'date_created': '2026-10-04T09:51:40.590273+00:00', 'doc_type_counts': {'text': 17}, 'total_elements': 17, 'raw_text_elements_size': 59317}}], 'failed_documents': [], 'validation_errors': []}, 'nv_ingest_status': {'extraction_completed': 1, 'document_wise_status': {'SLM_Future_AI.pdf': 'completed'}}}
HTTP Status: 200
{'state': 'FINISHED', 'result': {'message': 'Document upload job successfully completed.', 'total_documents': 1, 'documents_completed': 1, 'batches_completed': 1, 'documents': [{'document_name': 'SLM_Future_AI.pdf', 'metadata': {'filename': 'SLM_Future_AI.pdf'}, 'document_info': {'description': '', 'tags':

In [21]:
RAG_BASE = "http://rag-server:8081/v1"

import requests

question = "What is this document about?"

payload = {
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ],
    "use_knowledge_base": True,
    "collection_names": ["research_papers"],
    "enable_reranker": True,
    "enable_citations": True
}

In [24]:
with requests.post(
    f"{RAG_BASE}/generate",
    json=payload,
    stream=True,
    timeout=300
) as r:
    print("HTTP Status:", r.status_code)
    r.raise_for_status()

    for line in r.iter_lines(decode_unicode=True):
        if not line:
            continue

        if line.startswith("data: "):
            line = line[6:]

        if line == "[DONE]":
            break

        print(line)


# The garbage output is happening because code prints the entire raw SSE/JSON chunk.
# Those chunks can contain citations, metadata, token information, and large Base64-encoded images.

HTTP Status: 200
{"id":"f01cef77-7bfc-4228-93b6-dca8eb59e6ad","choices":[{"index":0,"message":{"role":"assistant","content":"We","reasoning_content":null},"delta":{"role":null,"content":"We","reasoning_content":null},"finish_reason":null}],"model":"nvidia/nemotron-3-super-120b-a12b","object":"chat.completion.chunk","created":1791109501,"usage":{"total_tokens":0,"prompt_tokens":0,"completion_tokens":0},"citations":{"total_results":10,"results":[{"document_id":"","content":"Small Language Models are the Future of Agentic AI\r\nPeter Belcak1 Greg Heinrich1 Shizhe Diao1 Yonggan Fu1 Xin Dong1\r\nSaurav Muralidharan1 Yingyan Celine Lin1,2 Pavlo Molchanov1\r\n1NVIDIA Research 2Georgia Institute of Technology\r\nagents-research@nvidia.com\r\nAbstract\r\nLarge language models (LLMs) are often praised for exhibiting near-human per\u0002formance on a wide range of tasks and valued for their ability to hold a general\r\nconversation. The rise of agentic AI systems is, however, ushering in a mass o

In [29]:
import json
import requests

RAG_BASE = "http://rag-server:8081/v1"

question = "What SLM Future AI describes"

payload = {
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ],
    "use_knowledge_base": True,
    "collection_names": ["research_papers"],
    "enable_reranker": True,
    "enable_citations": True
}

answer_parts = []
citations = None

with requests.post(
    f"{RAG_BASE}/generate",
    json=payload,
    stream=True,
    timeout=300
) as r:

    print("HTTP Status:", r.status_code)
    r.raise_for_status()

    for line in r.iter_lines(decode_unicode=True):

        if not line:
            continue

        if line.startswith("data: "):
            line = line[6:].strip()

        if line == "[DONE]":
            break

        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue

        # IMPORTANT:
        # Save the first chunk containing citations.
        if citations is None and data.get("citations"):
            citations = data["citations"]

        # Extract only the generated answer text
        choices = data.get("choices", [])

        if choices:
            delta = choices[0].get("delta", {})
            text = delta.get("content")

            if not text:
                message = choices[0].get("message", {})
                text = message.get("content", "")

            if text:
                answer_parts.append(text)

answer = "".join(answer_parts).strip()

print("\n" + "=" * 60)
print("RAG ANSWER")
print("=" * 60)
print(answer)

if citations:
    print("\n" + "=" * 60)
    print(f"CITATIONS: {len(citations.get('results', []))}")
    print("=" * 60)
else:
    print("\nCITATIONS: 0")

HTTP Status: 200

RAG ANSWER
We need answer: what SLM Future AI describes. Summarize: it's a paper arguing small language models are future of agentic AI. Provide description.

SLM Future AI is a research paper that argues small language models (SLMs) are the future of agentic AI. It contends that SLMs are sufficiently powerful, more operationally suitable, and far more economical than large language models (LLMs) for the repetitive, specialized tasks that dominate agentic systems. The paper supports this view with evidence of SLM performance, efficiency gains, flexibility, and modular deployment, and it outlines a conversion algorithm for migrating agentic applications from LLMs to SLMs, while acknowledging cases where heterogeneous systems that selectively use LLMs remain necessary.

CITATIONS: 10


In [30]:
import requests
import json
import base64
from IPython.display import display, Markdown, Image

RAG_BASE = "http://rag-server:8081/v1"

question = "What is this document about?"

payload = {
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ],
    "use_knowledge_base": True,
    "collection_names": ["research_papers"],
    "enable_reranker": True,
    "enable_citations": True
}

answer_parts = []
citations = None

with requests.post(
    f"{RAG_BASE}/generate",
    json=payload,
    stream=True,
    timeout=300
) as r:

    print("HTTP Status:", r.status_code)
    r.raise_for_status()

    for line in r.iter_lines(decode_unicode=True):

        if not line:
            continue

        if line.startswith("data: "):
            line = line[6:]

        if line == "[DONE]":
            break

        try:
            chunk = json.loads(line)
        except json.JSONDecodeError:
            continue

        # Save citations from the first chunk that contains them
        if citations is None and chunk.get("citations"):
            citations = chunk["citations"]

        choices = chunk.get("choices", [])

        if not choices:
            continue

        delta = choices[0].get("delta", {})
        text = delta.get("content")

        if text:
            answer_parts.append(text)
            print(text, end="", flush=True)

print("\n\n--- Complete answer ---")
answer = "".join(answer_parts)
print(answer)

HTTP Status: 200
We need to answer: "What is this document about?" The context includes multiple files: SLM_Future_AI (about small language models future of agentic AI), BLT-Paper (about Byte Latent Transformer). The question likely refers to the whole provided context? Probably they ask about the document (the context). We can say it's about SLMs and BLT. Provide summary.

The provided text consists of several research excerpts. The main focus is on two related topics:

1. **Small Language Models (SLMs) as the future of agentic AI** – a paper arguing that SLMs are sufficiently powerful, more suitable, and more economical for many tasks in agentic systems, with case studies (MetaGPT, Open Operator, Cradle) estimating the proportion of LLM queries that could be replaced by SLMs, and describing an LLM‑to‑SLM conversion algorithm.

2. **Byte Latent Transformer (BLT)** – a tokenizer‑free architecture that processes raw bytes into dynamically sized patches, showing improved inference effici